<a href="https://colab.research.google.com/github/memo124/Pr-ctica-de-Redes-Neuronales-Dise-o-de-Arquitecturas-Densas/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Práctica de Redes Neuronales: Diseño de Arquitecturas Densas
**Equipo y Roles:**
* **Integrante 1 (Especialista en Datos y Arquitecto de Red - Modelos A y B):** Guillermo Andrés Minero Alfaro
* **Integrante 2 (Arquitecto de Red - Modelo C y Analista de Métricas):** [Nombre del compañero]

---
## 1. Preparación y Datos
Se importan las librerías necesarias, se carga el dataset y se realiza el preprocesamiento esencial para redes neuronales (escalado y división).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# Cargar dataset de clasificación binaria
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

# Dividir en entrenamiento y prueba (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Normalizar/Escalar los datos (Crucial para que las redes neuronales converjan correctamente)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Dimensiones de entrenamiento: {X_train_scaled.shape}")
print(f"Dimensiones de prueba: {X_test_scaled.shape}")

## 2. Modelo A (El Minimalista)
**Justificación de Arquitectura:**
Para esta primera variante, se implementa una red neuronal muy simple con una única capa oculta de 8 neuronas. El objetivo es establecer un rendimiento base. Al tener pocos parámetros, esta red corre poco riesgo de sobreajuste (overfitting), pero podría sufrir de subajuste (underfitting) si el problema fuera altamente complejo. Se utiliza activación ReLU en la capa oculta por su eficiencia y Sigmoid en la salida para obtener una probabilidad binaria.

In [ ]:
# Construcción del Modelo A
model_A = Sequential([
    Dense(8, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dense(1, activation='sigmoid') # Clasificación binaria
])

# Compilación
model_A.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Entrenamiento
print("Entrenando Modelo A (Minimalista)...")
history_A = model_A.fit(
    X_train_scaled, y_train,
    validation_data=(X_test_scaled, y_test),
    epochs=100,
    batch_size=32,
    verbose=0 # verbose=0 para no saturar la pantalla de logs
)
print("Modelo A entrenado.")

## 3. Modelo B (El Profundo)
**Justificación de Arquitectura:**
En esta segunda variante, construimos una red profunda (Deep Learning) agregando múltiples capas ocultas (64, 128 y 64 neuronas) sin ningún tipo de regularización. El propósito de esta arquitectura sobredimensionada para este conjunto de datos es forzar intencionalmente a la red a memorizar los datos de entrenamiento para evidenciar el fenómeno de **Overfitting**.

In [ ]:
# Construcción del Modelo B
model_B = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dense(128, activation='relu'),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])

# Compilación
model_B.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Entrenamiento (aumentamos épocas para forzar el overfitting)
print("Entrenando Modelo B (Profundo)...")
history_B = model_B.fit(
    X_train_scaled, y_train,
    validation_data=(X_test_scaled, y_test),
    epochs=150,
    batch_size=32,
    verbose=0
)
print("Modelo B entrenado.")

## 4. Gráficas Iniciales y Análisis de Overfitting
A continuación graficamos las curvas de Aprendizaje (Loss) y Precisión (Accuracy) para comparar el Modelo A y el Modelo B.

In [ ]:
def plot_history(history_A, history_B, title_A, title_B):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # Loss Modelo A
    axes[0, 0].plot(history_A.history['loss'], label='Train Loss')
    axes[0, 0].plot(history_A.history['val_loss'], label='Val Loss')
    axes[0, 0].set_title(f'{title_A} - Loss')
    axes[0, 0].set_xlabel('Epochs')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].legend()

    # Accuracy Modelo A
    axes[0, 1].plot(history_A.history['accuracy'], label='Train Acc')
    axes[0, 1].plot(history_A.history['val_accuracy'], label='Val Acc')
    axes[0, 1].set_title(f'{title_A} - Accuracy')
    axes[0, 1].set_xlabel('Epochs')
    axes[0, 1].set_ylabel('Accuracy')
    axes[0, 1].legend()

    # Loss Modelo B
    axes[1, 0].plot(history_B.history['loss'], label='Train Loss')
    axes[1, 0].plot(history_B.history['val_loss'], label='Val Loss')
    axes[1, 0].set_title(f'{title_B} - Loss')
    axes[1, 0].set_xlabel('Epochs')
    axes[1, 0].set_ylabel('Loss')
    axes[1, 0].legend()

    # Accuracy Modelo B
    axes[1, 1].plot(history_B.history['accuracy'], label='Train Acc')
    axes[1, 1].plot(history_B.history['val_accuracy'], label='Val Acc')
    axes[1, 1].set_title(f'{title_B} - Accuracy')
    axes[1, 1].set_xlabel('Epochs')
    axes[1, 1].set_ylabel('Accuracy')
    axes[1, 1].legend()

    plt.tight_layout()
    plt.show()

plot_history(history_A, history_B, 'Modelo A (Minimalista)', 'Modelo B (Profundo)')

**Identificación visual del Overfitting:**
Al observar las gráficas del **Modelo B (Profundo)**, podemos identificar claramente el sobreajuste. A partir de la época ~15-20, el `Train Loss` continúa disminuyendo acercándose a cero, mientras que el `Val Loss` deja de mejorar y comienza a aumentar drásticamente. Esto indica que la red dejó de aprender patrones generalizables y comenzó a memorizar los datos de entrenamiento de forma exacta. El Modelo A, al tener menos capacidad, mantiene las curvas de entrenamiento y validación mucho más juntas a lo largo del tiempo.